# Salary Prediction — Live Kafka Streaming Test (VM)

Starts the real-time prediction stream (`src/streaming/prediction_stream.py`, PLAN.md §16), publishes one test request to `salary_requests`, and reads back the matching prediction from `salary_predictions` — all from this notebook, no separate terminal needed.

**Prerequisites:** Kafka broker running, all 5 topics created, and `models/best_salary_model` already exists (i.e. `run_training_pipeline.ipynb` has been run at least once). **Run cells top to bottom.**

## 1. Path and working directory

In [ ]:
import sys, os

PROJECT_ROOT = "/home/linuxu/project"  # adjust if this VM's checkout lives elsewhere

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

## 1a. Install missing Python packages (one-time)

If a later cell fails with `ModuleNotFoundError`, add `%pip install <package-name>` here and re-run from the top after restarting the kernel.

In [ ]:
%pip install python-dotenv confluent-kafka

## 2. `.env` check

In [ ]:
if not os.path.exists(".env"):
    import subprocess
    subprocess.run(["cp", ".env.example", ".env"])
    print("Created .env from .env.example.")

print(open(".env").read())

Confirm `KAFKA_BOOTSTRAP_SERVERS`, `KAFKA_REQUEST_TOPIC`, `KAFKA_PREDICTION_TOPIC`, `KAFKA_DEAD_LETTER_TOPIC` above match what you actually created on the VM (per PLAN.md §6 / the Lab3-style setup: `salary_requests`, `salary_predictions`, `salary_dead_letter`).

## 3. Spark session — force a fresh one, with Kafka support

In [ ]:
from pyspark.sql import SparkSession

try:
    SparkSession.builder.getOrCreate().stop()
    print("Stopped an existing Spark session.")
except Exception as exc:
    print("No existing session to stop (or stop failed) - continuing:", exc)

In [ ]:
from src.common.spark_session import get_spark_session
from config import settings

spark = get_spark_session(app_name="SalaryPredictionStream", with_kafka=True)

print("spark.driver.memory      =", spark.sparkContext.getConf().get("spark.driver.memory"))
print("KAFKA_BOOTSTRAP_SERVERS  =", settings.KAFKA_BOOTSTRAP_SERVERS)
print("KAFKA_REQUEST_TOPIC      =", settings.KAFKA_REQUEST_TOPIC)
print("KAFKA_PREDICTION_TOPIC   =", settings.KAFKA_PREDICTION_TOPIC)
print("MODEL_PATH               =", settings.MODEL_PATH)
print("MODEL_PATH exists?       =", os.path.exists(settings.MODEL_PATH))

## 4. Start the prediction stream (non-blocking)

This starts both streaming queries (predictions and dead letters) and returns immediately — it does **not** block the notebook, unlike the CLI entry point (`python -m src.streaming.prediction_stream`), which awaits termination forever. You're responsible for stopping the queries yourself in the cleanup cell at the bottom.

In [ ]:
from src.streaming.prediction_stream import build_streams

prediction_query, dead_letter_query = build_streams(spark)

print("prediction_query.isActive  =", prediction_query.isActive)
print("dead_letter_query.isActive =", dead_letter_query.isActive)

## 5. Publish a test request

Same pattern as Lab3's `confluent_kafka` producer.

In [ ]:
import json
import uuid
from confluent_kafka import Producer

producer = Producer({"bootstrap.servers": settings.KAFKA_BOOTSTRAP_SERVERS})

test_request = {
    "request_id": str(uuid.uuid4()),
    "Country": "Israel",
    "Age": "25-34 years old",
    "EdLevel": "Bachelor's degree",
    "Employment": "Employed, full-time",
    "RemoteWork": "Hybrid",
    "DevType": "Developer, back-end",
    "OrgSize": "100 to 499 employees",
    "Industry": "Information Services, IT, Software Development",
    "YearsCodePro": 5,
    "LanguageHaveWorkedWith": "Python;SQL",
    "DatabaseHaveWorkedWith": "PostgreSQL",
    "PlatformHaveWorkedWith": "AWS",
}

producer.produce(
    settings.KAFKA_REQUEST_TOPIC,
    key=test_request["request_id"],
    value=json.dumps(test_request),
)
producer.flush()
print("Published request_id:", test_request["request_id"])

## 6. Wait, then check the result

Give the stream a few seconds to pick up the request and publish a prediction before checking. Re-run this cell if `matches` comes back empty the first time.

In [ ]:
import time
from pyspark.sql import functions as F

time.sleep(15)

results = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", settings.KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", settings.KAFKA_PREDICTION_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
    .select(F.col("value").cast("string").alias("value"))
)

matches = [row["value"] for row in results.collect() if test_request["request_id"] in row["value"]]

if matches:
    print("Found matching prediction(s):")
    for match in matches:
        print(json.dumps(json.loads(match), indent=2))
else:
    print("No match yet for request_id", test_request["request_id"], "- wait a bit and re-run this cell.")
    print("All predictions currently on the topic:", results.count())

## 7. (Optional) Test the dead-letter path

Publishes a request with no `request_id` — should NOT appear on `salary_predictions`, and should instead show up on `salary_dead_letter`.

In [ ]:
bad_request = {"Country": "Israel", "YearsCodePro": 3}  # no request_id on purpose

producer.produce(settings.KAFKA_REQUEST_TOPIC, value=json.dumps(bad_request))
producer.flush()
print("Published a request with no request_id.")

time.sleep(15)

dead_letters = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", settings.KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", settings.KAFKA_DEAD_LETTER_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
    .select(F.col("value").cast("string"))
)
dead_letters.show(truncate=False)

## 8. Cleanup — stop the streaming queries

Run this when you're done testing, so the queries don't keep running in the background.

In [ ]:
prediction_query.stop()
dead_letter_query.stop()
print("Stopped both streaming queries.")